In [1]:
!pip install -q "psycopg[binary]"


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import psycopg

conn = psycopg.connect(
    host="localhost",
    port=5433,
    dbname="ai_workshop",
    user="postgres",
    password="kumar9"
)
print("Connected!")

In [3]:
with conn.cursor() as cur:
    cur.execute("SELECT * FROM developer")
    rows = cur.fetchall()

for row in rows:
    print(row)

(1, 'Alice Smith', 'Engineering', 28)
(2, 'Bob Jones', 'Data Science', 34)
(5, 'Evan Wright', 'DevOps', 32)
(6, 'Fiona Gallagher', 'Quality Assurance', 27)
(7, 'George Clark', 'Security', 41)
(8, 'Hannah Abbott', 'Engineering', 29)
(9, 'Ian Malcolm', 'Data Science', 38)
(10, 'Julia Roberts', 'Product', 35)
(11, 'Kevin Bacon', 'Backend', 45)
(12, 'Laura Croft', 'Frontend', 26)
(13, 'Michael Scott', 'Management', 42)
(14, 'Jim Halpert', 'Sales Tech', 31)
(15, 'Pam Beesly', 'Design', 30)
(16, 'Dwight Schrute', 'Security', 37)
(17, 'Stanley Hudson', 'Backend', 50)
(18, 'Angela Martin', 'Data Science', 39)
(19, 'Ryan Howard', 'Engineering', 24)
(20, 'Kelly Kapoor', 'Frontend', 28)
(21, 'Creed Bratton', 'Legacy Systems', 65)
(22, 'Toby Flenderson', 'HR Tech', 44)
(23, 'Oscar Martinez', 'Data Science', 36)
(24, 'Andy Bernard', 'Engineering', 33)
(25, 'Phyllis Vance', 'Support Tech', 48)
(3, 'Charlie Brown', 'Frontend', 25)
(26, 'Kiran', 'CSE', 21)


In [4]:
with conn.cursor() as cur:
    cur.execute(
        """
        INSERT INTO developer
            (name, department, age)
        VALUES (%s, %s, %s)
        """,
        ("Kiran", "CSE", 21)
    )

conn.commit()

print("Data inserted!")

Data inserted!


In [5]:
!pip install -U "google-genai>=1.66.0,<2.0.0"

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from google import genai
from google.genai import types

with open("api_key.txt", "r") as f:
    GEMINI_API_KEY = f.read().strip()

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client ready")

Gemini client ready


In [8]:
TEXT_MODEL = "gemini-3.5-flash-lite"

In [9]:
response = client.models.generate_content(
    model=TEXT_MODEL,
    contents="Explain what a database is in one simple sentence."
)
print(response.text)

A database is an organized collection of data stored electronically so it can be easily accessed, managed, and updated.


In [10]:
EMBEDDING_MODEL = "gemini-embedding-001"
def create_embedding(text):
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(output_dimensionality=768)
    )
    return result.embeddings[0].values
e = create_embedding("PostgreSQL is a database.")
print(len(e))      # 768
print(e[:5])

768
[-0.015443377, -0.0012037307, -0.005239699, -0.063788414, -0.026394403]


In [11]:
sentences = ["I love dogs.",
             "I like puppies.",
             "The stock market fell today."]
embeddings = [create_embedding(t) for t in sentences]
print(len(embeddings))       # 3
print(len(embeddings[0]))    # 768

3
768


In [12]:
text = "PostgreSQL is an open-source relational database."
embedding = create_embedding(text)
with conn.cursor() as cur:
    cur.execute(
        "INSERT INTO document (content, embedding) VALUES (%s, %s)",
        (text, embedding)
    )
conn.commit()
print("Document inserted!")

Document inserted!


In [26]:
conn.rollback()

In [46]:
conn.rollback()
print("Transaction reset")

Transaction reset


In [34]:
with conn.cursor() as cur:
    cur.execute("SELECT 1")
    print(cur.fetchone())

(1,)


In [35]:
with conn.cursor() as cur:
    cur.execute("SELECT content FROM document")
    for row in cur.fetchall():
        print(row)

('PostgreSQL is an open-source relational database.',)


In [36]:
documents = [
    "PostgreSQL is an open-source relational database.",
    "PostgreSQL can store and manage structured information.",
    "MongoDB is a document-oriented database.",
    "Python is a popular programming language used in AI.",
    "Machine learning allows computers to learn patterns from data."
]

In [37]:
with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE document RESTART IDENTITY")
conn.commit()

In [38]:
for text in documents:
    embedding = create_embedding(text)
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO document (content, embedding) VALUES (%s, %s)",
            (text, embedding)
        )
    conn.commit()

In [ ]:
CREATE EXTENSION IF NOT EXISTS vector;
SELECT extname FROM pg_extension;
-- 'vector' should be listed

In [44]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM document")
    print("Number of document:", cur.fetchone()[0])

Number of document: 5


In [47]:
query = "Which system can manage structured information?"
query_embedding = create_embedding(query)

with conn.cursor() as cur:
    cur.execute(
        """
        SELECT content, embedding <=> %s::vector AS distance
        FROM document
        ORDER BY embedding <=> %s::vector
        LIMIT 5
        """,
        (query_embedding, query_embedding)
    )
    results = cur.fetchall()

for content, distance in results:
    print(round(distance, 3), content)

0.188 PostgreSQL can store and manage structured information.
0.44 MongoDB is a document-oriented database.
0.445 PostgreSQL is an open-source relational database.
0.453 Machine learning allows computers to learn patterns from data.
0.477 Python is a popular programming language used in AI.


In [48]:
def semantic_search(query, top_k=3):
    query_embedding = create_embedding(query)
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT content, embedding <=> %s::vector AS distance
            FROM document
            ORDER BY embedding <=> %s::vector
            LIMIT %s
            """,
            (query_embedding, query_embedding, top_k)
        )
        return cur.fetchall()
for content, distance in semantic_search("Which system can manage structured information?"):
    print(round(distance, 3), content)

0.188 PostgreSQL can store and manage structured information.
0.44 MongoDB is a document-oriented database.
0.445 PostgreSQL is an open-source relational database.


In [49]:
for q in ["Which database can store information?",
          "Which programming language is popular in AI?",
          "How can computers learn from data?"]:
    print("QUESTION:", q)
    for content, distance in semantic_search(q):
        print(f"  {distance:.3f}  {content}")

QUESTION: Which database can store information?
  0.278  PostgreSQL can store and manage structured information.
  0.391  MongoDB is a document-oriented database.
  0.410  PostgreSQL is an open-source relational database.
QUESTION: Which programming language is popular in AI?
  0.159  Python is a popular programming language used in AI.
  0.429  Machine learning allows computers to learn patterns from data.
  0.515  PostgreSQL can store and manage structured information.
QUESTION: How can computers learn from data?
  0.179  Machine learning allows computers to learn patterns from data.
  0.415  Python is a popular programming language used in AI.
  0.472  PostgreSQL can store and manage structured information.


In [50]:
def ask(question, top_k=3):
    found = semantic_search(question, top_k)
    context = "\n".join(f"{i + 1}. {c}" for i, (c, d) in enumerate(found))
    prompt = f"""Answer the question using ONLY the information below.
If the answer is not there, say "I don't know".
Information:
{context}
Question: {question}"""
    return client.models.generate_content(model=TEXT_MODEL, contents=prompt).text
print(ask("Which system can manage structured information?"))
print(ask("Is the canteen open on Sunday?"))     # should say "I don't know"

PostgreSQL
I don't know
